In [17]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

from itertools import product
import pandas as pd
import random
import os
import time

In [18]:
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

driver.get("http://103.233.100.26:8080/")
driver.maximize_window()

time.sleep(5)

In [19]:
os.makedirs("dataset/images", exist_ok=True)

In [20]:
buttons = driver.find_elements(By.TAG_NAME, "button")

print("Buttons ditemukan:")

for i, btn in enumerate(buttons):
    print(i, btn.text)

Buttons ditemukan:
0 Front Left Door
1 Front Right Door
2 Rear Left Door
3 Rear Right Door
4 Hood


In [21]:
canvas = driver.find_element(By.TAG_NAME, "canvas")

In [22]:
def reset_scene():
    global current_state
    
    driver.refresh()
    time.sleep(8)
    
    buttons = driver.find_elements(By.TAG_NAME, "button")
    canvas = driver.find_element(By.TAG_NAME, "canvas")
    
    # After refresh, everything is closed by default
    current_state = [0, 0, 0, 0, 0]
    
    print(f"Reset complete - current_state: {current_state}")
    
    return buttons, canvas

In [23]:
current_state = [0,0,0,0,0]

In [24]:
def set_state(target_state):

    global current_state

    buttons = driver.find_elements(By.TAG_NAME, "button")

    for i in range(5):

        if current_state[i] != target_state[i]:

            buttons[i].click()

            time.sleep(0.7)

            current_state[i] = target_state[i]

In [25]:
def rotate_car(offset):
    canvas = driver.find_element(By.TAG_NAME, "canvas")

    w = canvas.size["width"]
    h = canvas.size["height"]

    ActionChains(driver)\
        .move_to_element_with_offset(
            canvas,
            -w//2 + 50,
            -h//2 + 50
        )\
        .click_and_hold()\
        .move_by_offset(offset, 0)\
        .pause(0.2)\
        .release()\
        .perform()

    time.sleep(1)

In [26]:
def save_image():

    global counter

    canvas = driver.find_element(By.TAG_NAME, "canvas")

    filename = f"img_{counter:05d}.png"

    path = os.path.join(
        "dataset/images",
        filename
    )

    canvas.screenshot(path)

    counter += 1

    return filename

In [27]:
labels = []

In [28]:
def save_label(filename, state):

    labels.append([
        filename,
        state[0],
        state[1],
        state[2],
        state[3],
        state[4]
    ])

In [29]:
states = list(product([0,1], repeat=5))

print(len(states))

32


In [30]:
# views = {
#     "front_right": 0,
#     "rear_right": 250,
#     "rear_left": 500,
#     "front_left": 750
# }

In [31]:
canvas = driver.find_element(By.TAG_NAME, "canvas")

print(canvas.size)
print(canvas.location)

{'height': 701, 'width': 1366}
{'x': 171, 'y': 88}


In [32]:
import os
import re

files = os.listdir("dataset/images")

numbers = []

for f in files:

    m = re.match(
        r"img_(\d+)\.png",
        f
    )

    if m:
        numbers.append(
            int(m.group(1))
        )

counter = max(numbers) + 1

print(
    f"Starting from {counter}"
)

Starting from 1280


In [33]:
labels = []

for state in states:

    for variation in range(5):

        buttons, canvas = reset_scene()

        set_state(state)

        angle = random.randint(0, 1000)

        rotate_car(angle)

        filename = save_image()

        save_label(filename, state)

        print(
            filename,
            state,
            f"angle={angle}"
        )

Reset complete - current_state: [0, 0, 0, 0, 0]
img_01280.png (0, 0, 0, 0, 0) angle=475
Reset complete - current_state: [0, 0, 0, 0, 0]
img_01281.png (0, 0, 0, 0, 0) angle=463
Reset complete - current_state: [0, 0, 0, 0, 0]
img_01282.png (0, 0, 0, 0, 0) angle=450
Reset complete - current_state: [0, 0, 0, 0, 0]
img_01283.png (0, 0, 0, 0, 0) angle=176
Reset complete - current_state: [0, 0, 0, 0, 0]
img_01284.png (0, 0, 0, 0, 0) angle=168
Reset complete - current_state: [0, 0, 0, 0, 0]
img_01285.png (0, 0, 0, 0, 1) angle=107
Reset complete - current_state: [0, 0, 0, 0, 0]
img_01286.png (0, 0, 0, 0, 1) angle=673
Reset complete - current_state: [0, 0, 0, 0, 0]
img_01287.png (0, 0, 0, 0, 1) angle=220
Reset complete - current_state: [0, 0, 0, 0, 0]
img_01288.png (0, 0, 0, 0, 1) angle=951
Reset complete - current_state: [0, 0, 0, 0, 0]
img_01289.png (0, 0, 0, 0, 1) angle=387
Reset complete - current_state: [0, 0, 0, 0, 0]
img_01290.png (0, 0, 0, 1, 0) angle=535
Reset complete - current_state: 

In [34]:
import pandas as pd

old_df = pd.read_csv(
    "dataset/labels.csv"
)

new_df = pd.DataFrame(
    labels,
    columns=[
        "filename",
        "fl",
        "fr",
        "rl",
        "rr",
        "hood"
    ]
)

df = pd.concat(
    [old_df, new_df],
    ignore_index=True
)

df.to_csv(
    "dataset/labels.csv",
    index=False
)